# 03 - Pass 3: coherency check (text only)

`Qwen/Qwen3.8-Flash-Next` re-reads the pass-2 transcription from text alone (question, reference working, pass 1, pass 2; the question diagram is attached, the student photo is not) and marks places where a character may have been misread as `{{likely reading / pass-2 reading}}`. Prompt: `resources/prompts/pass3_coherency_v4.txt`. The reasoning text is kept in the records.

A pass-3 text that edits anything outside its marks is ignored later (`SILENT_EDIT_POLICY = 'fallback_pass2'`).

**Reads** `work/pass1`, `work/pass2`.  **Writes** `work/pass3/{records/,pass3_transcriptions.json,results.csv}`.

**Default: smoke test on 5 rows.** Set `LIMIT = None` in the second cell for the full run.

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, runner
LIMIT = 5         # SMOKE TEST: first 5 test rows only. Set LIMIT = None for the full 3,498 rows (finished rows are kept and skipped, so the full run resumes from the smoke test).
print('repo:', REPO, '| data:', common.DATA, '| work:', common.WORK)

In [ ]:
from misc_pipeline import pass3
jobs = pass3.build_jobs(LIMIT)
print(len(jobs), 'rows | with question diagram:', sum(1 for j in jobs if j['images']))

In [ ]:
summary = runner.run_stage('pass3', jobs, common.STAGE_DIRS['pass3'], pass3.parse, label_images=True)
print(summary)

In [ ]:
exp = pass3.export(jobs); print(exp)

## Free the GPUs
The stage already closes its model when it ends. Run this cell after an interrupted run or an error, or before starting the next notebook in the same kernel.

In [ ]:
import gc
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU memory in use (GB):', [round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(torch.cuda.device_count())])
except ImportError:
    pass
# If memory is still held, restart the kernel (Kernel > Restart); vLLM worker processes then exit.

## Optional: delete the downloaded model
The weights stay in the Hugging Face cache (`HF_HOME`) and are re-downloaded on the next use. Set `DELETE_MODEL = True` to remove `Qwen/Qwen3.8-Flash-Next` from the cache now. Keep it if a later notebook uses the same model (notebooks 02 and 03 share one model, as do 04 and 05).

In [ ]:
DELETE_MODEL = False
MODEL = 'Qwen/Qwen3.8-Flash-Next'
if DELETE_MODEL:
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    hashes = [r.commit_hash for repo in info.repos if repo.repo_id == MODEL for r in repo.revisions]
    if hashes:
        strategy = info.delete_revisions(*hashes); print('freeing %.1f GB' % (strategy.expected_freed_size / 1e9)); strategy.execute()
    else:
        print(MODEL, 'is not in the cache')
else:
    print('model kept in the cache')